# Gamma Regression Explorer

This notebook compares two display models over `volt in [0, 1]`:

- Regression model: `lum_reg = volt**gamma_reg`
- Gain/offset model: `lum = (volt_gain * (volt + volt_offset))**2.4`

`gamma_reg` is fit to minimize mean squared error between the two curves.


In [1]:
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from IPython.display import display


In [2]:
volt = np.linspace(0.0, 1.0, 1001, dtype=np.float64)


def model_luminance(voltages: np.ndarray, volt_gain: float, volt_offset: float) -> np.ndarray:
    # Keep model output in [0, 1] for direct comparison to the regression model.
    drive = np.clip(volt_gain * (voltages + volt_offset), 0.0, 1.0)
    return np.power(drive, 2.4)


def mse_for_gamma(gamma: float, voltages: np.ndarray, lum_target: np.ndarray) -> float:
    lum_reg = np.power(voltages, gamma)
    return float(np.mean((lum_reg - lum_target) ** 2))


def fit_gamma_reg(voltages: np.ndarray, lum_target: np.ndarray, lo: float = 0.05, hi: float = 6.0, iterations: int = 80) -> float:
    # Golden-section search on one variable: gamma_reg.
    inv_phi = (np.sqrt(5.0) - 1.0) / 2.0
    c = hi - inv_phi * (hi - lo)
    d = lo + inv_phi * (hi - lo)
    fc = mse_for_gamma(c, voltages, lum_target)
    fd = mse_for_gamma(d, voltages, lum_target)

    for _ in range(iterations):
        if fc < fd:
            hi = d
            d = c
            fd = fc
            c = hi - inv_phi * (hi - lo)
            fc = mse_for_gamma(c, voltages, lum_target)
        else:
            lo = c
            c = d
            fc = fd
            d = lo + inv_phi * (hi - lo)
            fd = mse_for_gamma(d, voltages, lum_target)

    return float((lo + hi) * 0.5)


def make_figure(lum: np.ndarray, lum_reg: np.ndarray, abs_diff: np.ndarray, title: str) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.12,
        subplot_titles=('Luminance Curves', 'Absolute Difference'),
    )

    fig.add_trace(
        go.Scatter(x=volt, y=lum, mode='lines', name='lum (2.4 model, as modified)', line=dict(color='#1f77b4', width=3)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=volt, y=lum_reg, mode='lines', name='lum_reg (fitted gamma)', line=dict(color='#d62728', width=3, dash='dash')),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=volt,
            y=abs_diff,
            mode='lines',
            name='|difference|',
            line=dict(color='#2ca02c', width=2),
            fill='tozeroy',
            fillcolor='rgba(44,160,44,0.18)',
            showlegend=False,
        ),
        row=1, col=2,
    )

    fig.update_xaxes(title_text='volt', range=[0, 1], row=1, col=1)
    fig.update_xaxes(title_text='volt', range=[0, 1], row=1, col=2)
    fig.update_yaxes(title_text='luminance', range=[0, 1], automargin=False, row=1, col=1)

    abs_max = float(np.max(abs_diff))
    abs_upper = max(abs_max * 1.05, 1e-20)
    tick_vals = np.linspace(0.0, abs_upper, 5)
    tick_text = [f'{v:.1e}' for v in tick_vals]
    fig.update_yaxes(
        title_text='abs error',
        range=[0.0, abs_upper],
        tickmode='array',
        tickvals=tick_vals,
        ticktext=tick_text,
        automargin=False,
        row=1,
        col=2,
    )

    fig.update_layout(
        title=title,
        width=1460,
        height=430,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='left', x=0.0),
        margin=dict(l=100, r=30, t=90, b=60),
    )

    return fig


gain_slider = widgets.FloatSlider(
    min=0.85,
    max=1.15,
    step=0.01,
    value=1.0,
    continuous_update=True,
    readout=False,
    layout=widgets.Layout(width='380px'),
)

offset_slider = widgets.FloatSlider(
    min=-0.1,
    max=0.1,
    step=0.01,
    value=0.0,
    continuous_update=True,
    readout=False,
    layout=widgets.Layout(width='380px'),
)

gain_value = widgets.HTML(layout=widgets.Layout(width='70px'))
offset_value = widgets.HTML(layout=widgets.Layout(width='70px'))

gain_reset = widgets.Button(
    description='Reset',
    tooltip='Reset volt_gain to 1.0',
    layout=widgets.Layout(width='58px', height='30px'),
)

offset_reset = widgets.Button(
    description='Reset',
    tooltip='Reset volt_offset to 0.0',
    layout=widgets.Layout(width='58px', height='30px'),
)

gain_row = widgets.HBox(
    [
        widgets.HTML('<span style="font-weight:600;">volt_gain</span>', layout=widgets.Layout(width='90px')),
        widgets.HTML('<span style="color:#8a8a8a;">0.85</span>', layout=widgets.Layout(width='35px')),
        gain_slider,
        widgets.HTML('<span style="color:#8a8a8a;">1.15</span>', layout=widgets.Layout(width='35px')),
        gain_value,
        gain_reset,
    ],
    layout=widgets.Layout(align_items='center'),
)

offset_row = widgets.HBox(
    [
        widgets.HTML('<span style="font-weight:600;">volt_offset</span>', layout=widgets.Layout(width='90px')),
        widgets.HTML('<span style="color:#8a8a8a;">-0.10</span>', layout=widgets.Layout(width='35px')),
        offset_slider,
        widgets.HTML('<span style="color:#8a8a8a;">0.10</span>', layout=widgets.Layout(width='35px')),
        offset_value,
        offset_reset,
    ],
    layout=widgets.Layout(align_items='center'),
)

stats = widgets.HTML()
plot_output = widgets.Output()


def refresh(*_):
    volt_gain = float(gain_slider.value)
    volt_offset = float(offset_slider.value)

    gain_value.value = f'<span>{volt_gain:.2f}</span>'
    offset_value.value = f'<span>{volt_offset:.2f}</span>'

    lum = model_luminance(volt, volt_gain=volt_gain, volt_offset=volt_offset)
    gamma_reg = fit_gamma_reg(volt, lum)
    lum_reg = np.power(volt, gamma_reg)

    abs_diff = np.abs(lum - lum_reg)
    mse = float(np.mean((lum - lum_reg) ** 2))
    rmse = float(np.sqrt(mse))
    max_abs = float(np.max(abs_diff))

    title = (
        f'Best-fit gamma_reg = {gamma_reg:.6f} for ' +
        f'volt_gain={volt_gain:.2f}, volt_offset={volt_offset:.2f}'
    )

    with plot_output:
        plot_output.clear_output(wait=True)
        make_figure(lum, lum_reg, abs_diff, title).show()

    stats.value = (
        f"<b>Regression Results</b><br>"
        f"gamma_reg = {gamma_reg:.6f}<br>"
        f"MSE = {mse:.8f}<br>"
        f"RMSE = {rmse:.8f}<br>"
        f"Max |difference| = {max_abs:.8f}"
    )


def on_gain_reset(_):
    gain_slider.value = 1.0


def on_offset_reset(_):
    offset_slider.value = 0.0


gain_reset.on_click(on_gain_reset)
offset_reset.on_click(on_offset_reset)
gain_slider.observe(refresh, names='value')
offset_slider.observe(refresh, names='value')

controls = widgets.VBox([gain_row, offset_row])

display(controls)
display(stats)
display(plot_output)

refresh()


HTML(value='')

Output()